In [1]:
from dotenv import load_dotenv
load_dotenv()
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain.chat_models import init_chat_model
from langchain_mistralai import MistralAIEmbeddings
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/home/b1swas/RAG/RagProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_32861/1074941440.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader=PyPDFLoader("/home/b1swas/RAG/RagProject/src/Agentic_RAG/medical_report.pdf")
docs=loader.load()
len(docs)

9

In [3]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splitted_docs=splitter.split_documents(docs)
len(splitted_docs)

26

In [4]:
embedding_model=MistralAIEmbeddings()
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embedding_model,
)


In [5]:
same_record=vector_store.similarity_search("Patient name")
same_record[0].page_content

'Report Status    \nFemale\n27 Years:\n:\n:\n:\nAge\nGender\nReported        \nP\n9/7/2025   4:56:00PM\nDR NITIN NAHAR\n474764803\nMs. NIKITA  CHUDHARY:\n:\n:\n:\n:\nName        \nLab No.    \nRef By \nCollected       \nA/c Status Final\n10/7/2025  6:31:50PM\n:Collected at            :Processed at             BHOPAL CC-82\nMr Rachel V John Pata So Vitus John Mig 26 \nGraund,Indrapuri, Phone: 8770817968\n \nLPL-NATIONAL REFERENCE LAB\nNational Reference laboratory, Block E, \nSector 18, Rohini, New Delhi -110085\nTest Report      \nTest Name Results Units Bio. Ref. Interval\nHLA - B27\n(Flow Cytometry)\nHLA-B27, Disease Association   Negative\nInterpretation\n ------------------------------------------------------------\n| RESULT         |         REMARKS                           |\n|                |                                           |\n|----------------|-------------------------------------------|\n| Negative       | No expression of HLA-B27                  |'

In [9]:
@tool
def retriever_tool(query:str):
    """This tool can help you to retrieve the relevant data of the PDF Documents, and then answer according to the details on medical report"""
    print("Tool called",query)
    docs=vector_store.similarity_search(query=query,k=4)
    context=""
    for doc in docs:
        context+=doc.page_content+"\n\n"
    return context

In [16]:
result=retriever_tool.invoke({"query":"Patient Name"})
print(result)

Tool called Patient Name
NKIHJAFMDKOGFAAMFNOIHMIKHEJFEKFBJFFKBKOFOOBIOCNKDJPHNEKLJ
NMDBPJFCEHNJJDCLCDBFGPNFIDFKPNJHKEPLELPLNKCKIGEOCIGKHMEJP
ICIKOLFLGKCBMMCMPNBDAJEEINMEFCBBJMFEBFOEONDMKPFKEPCIGKAMG
ACICGJFCPLDDJOEMHHGHBPDEIEOPPFOFAOHPBLNOOLJBIKNJPKONHDICL
MNNNNNEHKHANBCCJOCFLHILHJBAHFHADLKPFCMNLMKJCLFNOAHFHAHIKL
APBBBPAPBEKAEJBCPMBKCFOFAGFCHHCAONFEPKPGOLMHNLNNEDFCGBKHH
HHHHHHHPHHHPHPPPPPPPPPHPPHPPPPPPPPPPHPPPPHHHPHPHHHHPPHPHP
IMPORTANT INSTRUCTIONS
ŸTest results released pertain to the specimen submitted .ŸAll test results are dependent on the quality of the sample received by the Laboratory . 
ŸLaboratory investigations are only a tool to facilitate in arriving at a diagnosis and should be clinically correlated by the Referring Physician .ŸReport 
delivery may be delayed due to unforeseen circumstances. Inconvenience is regretted .ŸCertain tests may require further testing at additional cost

Report Status    
Female
27 Years:
:
:
:
Age
Gender
Reported        
P
9/7/2025   4:56:00PM


In [23]:
llm_model=init_chat_model(
    model="mistral-small-latest"
)

In [22]:
System_Prompt="""
you are a helpfull assistant that answers questions using retrived context.
Always use the "retriver_tool" for questions requiring external knowledge
"""
agent=create_agent(
    model=llm_model,
    tools=[retriever_tool],
    system_prompt=System_Prompt,
)

In [24]:
query="what is ai?"

response=agent.invoke({"messages":[{'role':'user',"content":query}]})

HTTPStatusError: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}

In [ ]:
result=response['messages'][-1].content
print(result)

**Artificial Intelligence (AI)** is a field of computer science that focuses on creating systems or machines capable of performing tasks that typically require human intelligence. These tasks include:

1. **Learning**: AI systems can improve their performance over time by analyzing data and recognizing patterns (e.g., machine learning).
2. **Reasoning**: AI can make decisions or solve problems based on logical rules or algorithms.
3. **Problem-Solving**: AI can analyze complex situations and propose solutions (e.g., game-playing AI like AlphaGo).
4. **Perception**: AI can interpret sensory data, such as images (computer vision) or speech (natural language processing).
5. **Language Understanding**: AI can understand and generate human language (e.g., chatbots, translation tools).
6. **Automation**: AI can perform repetitive tasks without human intervention (e.g., robotic process automation).

AI can be categorized into:
- **Narrow AI (Weak AI)**: Designed for specific tasks (e.g., virt